# Stage 3/4 Evaluation and Diagnostics

Evaluation-only notebook for the completed six Stage 3 runs and six legacy Stage 4 runs. It does not rerun Stage 1 and does not modify trained checkpoints. The primary semantic metric throughout is normalized full recall-curve AUC over `k / N`, where chance is approximately 0.5 and perfect retrieval is 1.0.


In [ ]:
from __future__ import annotations

from pathlib import Path
import csv
import hashlib
import json
import math
import os
import platform
import random
import shutil
import subprocess
import sys
import time
from collections import Counter, defaultdict
from dataclasses import dataclass, asdict
from typing import Any

print("Python:", sys.version)
print("Platform:", platform.platform())

# Match notebooks 5/6: clone/update the configured branch into Colab and use
# Drive for completed inputs plus evaluation outputs.
REPO_URL = os.environ.get("NEUROVLM_REPO_URL", "https://github.com/neurovlm/neurovlm.git")
REPO_BRANCH = os.environ.get("NEUROVLM_REPO_BRANCH", "neurovlm_gnn")
REPO_DIR = Path(os.environ.get("NEUROVLM_REPO_DIR", "/content/neurovlm_gnn"))
DRIVE_ROOT = Path(os.environ.get("NEUROVLM_DRIVE_ROOT", "/content/drive/MyDrive/neurovlm"))
INSTALL_DEPENDENCIES = os.environ.get("NEUROVLM_INSTALL_DEPS", "1") == "1"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")


def run_cmd(cmd, cwd=None, *, check=True):
    print("$", " ".join(map(str, cmd)))
    result = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.returncode != 0:
        if result.stderr.strip():
            print(result.stderr.strip())
        if check:
            raise RuntimeError(f"Command failed ({result.returncode}): {' '.join(map(str, cmd))}")
    return result


if not REPO_DIR.exists():
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    run_cmd(["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)])
else:
    if not (REPO_DIR / ".git").exists():
        raise RuntimeError(
            f"{REPO_DIR} exists but is not a git checkout. Set NEUROVLM_REPO_DIR to a clean path "
            "or remove that folder, then rerun this cell."
        )
    run_cmd(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH])
    checkout = run_cmd(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], check=False)
    if checkout.returncode != 0:
        run_cmd(["git", "-C", str(REPO_DIR), "checkout", "-B", REPO_BRANCH, f"origin/{REPO_BRANCH}"])
    run_cmd(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", REPO_BRANCH])

os.chdir(REPO_DIR)

if INSTALL_DEPENDENCIES:
    run_cmd([sys.executable, "-m", "pip", "install", "-q", "nilearn", "nibabel", "huggingface-hub", "safetensors", "adapters", "transformers", "pyarrow", "matplotlib", "pandas", "scikit-learn", "tqdm", "umap-learn"])
    run_cmd([sys.executable, "-m", "pip", "install", "-q", "-e", ".[viz,notebook,metrics]"])

sys.path.insert(0, str(REPO_DIR / "experiments" / "3dcnn"))
sys.path.insert(0, str(REPO_DIR / "src"))
sys.path.insert(0, str(REPO_DIR))

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader

try:
    import matplotlib.pyplot as plt
except Exception:
    plt = None

from atlas_free_cnn.training.datasets import UnifiedMapTextDataset
from atlas_free_cnn.training.model_wrappers import (
    build_brain_encoder,
    build_cnn_autoencoder,
    build_generative_text_to_ae_latent,
    build_text_projection,
    build_text_to_brain_projection,
    load_autoencoder_checkpoint,
)
from atlas_free_cnn.evaluation.generation_metrics import generation_metrics
from neurovlm.metrics import bidirectional_retrieval_metrics, recall_curve, normalized_recall_curve_auc, retrieval_ranks

print("Working directory:", os.getcwd())
print("Repo branch:", run_cmd(["git", "branch", "--show-current"], cwd=REPO_DIR, check=False).stdout.strip())
print("Drive root:", DRIVE_ROOT)
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)


In [ ]:
# Hugging Face inputs and output layout. This mirrors notebooks 5/6.
def split_dir_has_jsonl(path: Path) -> bool:
    return all((path / name).exists() for name in ["train.jsonl", "val.jsonl", "test.jsonl"])


HF_DATASET_REPO = os.environ.get("NEUROVLM_ATLAS_FREE_HF_REPO", "neurovlm/atlas_free_cnn_dataset")
LOCAL_UNIFIED_CACHE_DIR = REPO_DIR / "experiments/3dcnn/atlas_free_cnn/cache/unified_jsonl_rebuild"
LOCAL_SPLIT_DIR = LOCAL_UNIFIED_CACHE_DIR / "splits"
LOCAL_PACK_DIR = REPO_DIR / "experiments/3dcnn/atlas_free_cnn/cache/hf_atlas_free_cnn_rebuild"
LOCAL_TEXT_CACHE_DIR = REPO_DIR / "experiments/3dcnn/atlas_free_cnn/cache/text_embeddings"


def hf_download_first_available(filenames, local_dir: Path) -> Path:
    from huggingface_hub import hf_hub_download
    local_dir.mkdir(parents=True, exist_ok=True)
    errors = []
    for filename in filenames:
        try:
            path = hf_hub_download(
                repo_id=HF_DATASET_REPO,
                repo_type="dataset",
                filename=filename,
                local_dir=str(local_dir),
                local_dir_use_symlinks=False,
            )
            return Path(path)
        except Exception as exc:
            errors.append(f"{filename}: {exc}")
    raise FileNotFoundError("Could not download any candidate from HF:\n" + "\n".join(errors))


def ensure_hf_unified_splits() -> Path:
    print(f"Downloading atlas-free CNN split JSONLs from Hugging Face: {HF_DATASET_REPO}")
    LOCAL_SPLIT_DIR.mkdir(parents=True, exist_ok=True)
    for split in ["train", "val", "test"]:
        downloaded = hf_download_first_available(
            [f"{split}.jsonl", f"splits/{split}.jsonl", f"unified_jsonl_rebuild/splits/{split}.jsonl"],
            LOCAL_UNIFIED_CACHE_DIR,
        )
        target = LOCAL_SPLIT_DIR / f"{split}.jsonl"
        if downloaded.resolve() != target.resolve():
            shutil.copy2(downloaded, target)
    for name in ["train_map_ids.json", "val_map_ids.json", "test_map_ids.json"]:
        try:
            downloaded = hf_download_first_available(
                [name, f"splits/{name}", f"unified_jsonl_rebuild/splits/{name}"],
                LOCAL_UNIFIED_CACHE_DIR,
            )
            target = LOCAL_SPLIT_DIR / name
            if downloaded.resolve() != target.resolve():
                shutil.copy2(downloaded, target)
        except Exception as exc:
            print(f"Optional split sidecar not downloaded ({name}): {exc}")
    try:
        downloaded_volume = hf_download_first_available(
            ["atlas_free_cnn_volumes.pt", "hf_atlas_free_cnn/atlas_free_cnn_volumes.pt", "hf_atlas_free_cnn_rebuild/atlas_free_cnn_volumes.pt"],
            LOCAL_PACK_DIR,
        )
        target_volume = LOCAL_PACK_DIR / "atlas_free_cnn_volumes.pt"
        if downloaded_volume.resolve() != target_volume.resolve():
            try:
                if target_volume.exists() or target_volume.is_symlink():
                    target_volume.unlink()
                os.symlink(downloaded_volume, target_volume)
            except Exception:
                shutil.copy2(downloaded_volume, target_volume)
        print("Volume tensor available at:", target_volume)
    except Exception as exc:
        print("WARNING: split JSONLs downloaded, but volume tensor was not prepared:", exc)
        print("Evaluation will fail unless tensor_path values inside JSONL resolve to an accessible tensor file.")
    return LOCAL_SPLIT_DIR


def discover_unified_split_dir() -> Path:
    override = os.environ.get("NEUROVLM_UNIFIED_SPLIT_DIR", "").strip()
    if override:
        override_path = Path(override).expanduser()
        if not split_dir_has_jsonl(override_path):
            raise FileNotFoundError(
                "NEUROVLM_UNIFIED_SPLIT_DIR was set, but it does not contain train.jsonl, val.jsonl, and test.jsonl: "
                f"{override_path}"
            )
        return override_path
    try:
        hf_split_dir = ensure_hf_unified_splits()
        if split_dir_has_jsonl(hf_split_dir):
            return hf_split_dir
    except Exception as exc:
        hf_error = exc
    else:
        hf_error = None
    raise FileNotFoundError(
        "Could not download unified split JSONLs from Hugging Face. Expected root files train.jsonl, val.jsonl, and test.jsonl.\n"
        f"HF dataset repo tried: {HF_DATASET_REPO}\nHF fallback error: {hf_error}"
    )


def discover_text_embedding_cache() -> Path:
    override = os.environ.get("NEUROVLM_TEXT_EMBEDDING_CACHE", "").strip()
    if override:
        override_path = Path(override).expanduser()
        if not override_path.exists():
            raise FileNotFoundError(f"NEUROVLM_TEXT_EMBEDDING_CACHE does not exist: {override_path}")
        return override_path
    try:
        downloaded = hf_download_first_available(
            [
                "specter_text_cache.pt",
                "text_embeddings/specter_text_cache.pt",
                "cache/text_embeddings/specter_text_cache.pt",
            ],
            LOCAL_TEXT_CACHE_DIR,
        )
        target = LOCAL_TEXT_CACHE_DIR / "specter_text_cache.pt"
        target.parent.mkdir(parents=True, exist_ok=True)
        if downloaded.resolve() != target.resolve():
            shutil.copy2(downloaded, target)
        return target
    except Exception as exc:
        hf_error = exc
    raise FileNotFoundError(
        "Could not download SPECTER text embedding cache from Hugging Face. "
        "Set NEUROVLM_TEXT_EMBEDDING_CACHE only if you need an explicit override.\n"
        f"HF dataset repo tried: {HF_DATASET_REPO}\nHF fallback error: {hf_error}"
    )


def discover_text_registry() -> Path | None:
    override = os.environ.get("NEUROVLM_TEXT_REGISTRY", "").strip()
    if override:
        override_path = Path(override).expanduser()
        if not override_path.exists():
            raise FileNotFoundError(f"NEUROVLM_TEXT_REGISTRY does not exist: {override_path}")
        return override_path
    try:
        downloaded = hf_download_first_available(
            ["text_registry.jsonl", "unified_jsonl/text_registry.jsonl", "rebuild_sources_pubmed_nilearn/text_registry.jsonl"],
            REPO_DIR / "experiments/3dcnn/atlas_free_cnn/cache/text_registry",
        )
        return downloaded
    except Exception as exc:
        print("Optional text registry was not downloaded from Hugging Face:", exc)
        return None


UNIFIED_SPLIT_DIR = discover_unified_split_dir()
TRAIN_JSONL = str(UNIFIED_SPLIT_DIR / "train.jsonl")
VAL_JSONL = str(UNIFIED_SPLIT_DIR / "val.jsonl")
TEST_JSONL = str(UNIFIED_SPLIT_DIR / "test.jsonl")
TEXT_EMBEDDING_CACHE = discover_text_embedding_cache()
TEXT_REGISTRY = discover_text_registry()
print("Unified split dir:", UNIFIED_SPLIT_DIR)
print("Text embedding cache:", TEXT_EMBEDDING_CACHE)
print("Text registry:", TEXT_REGISTRY or "not available")

STAMP = time.strftime("%Y%m%d_%H%M%S")
DEFAULT_EVAL_ROOT = DRIVE_ROOT / f"stage3_stage4_evaluation_{STAMP}" if "DRIVE_ROOT" in globals() else Path(f"stage3_stage4_evaluation_{STAMP}")
EVAL_ROOT = Path(os.environ.get("NEUROVLM_STAGE3_STAGE4_EVAL_ROOT", str(DEFAULT_EVAL_ROOT))).expanduser()
SUBDIRS = {
    "metadata": "00_metadata",
    "stage3_auc": "01_stage3_auc",
    "stage4_eval": "02_existing_stage4_checkpoint_eval",
    "text_cache": "03_text_cache_audit",
    "pairing": "04_pairing_audit",
    "architecture": "05_architecture_audit",
    "latent": "06_latent_diagnostics",
    "baselines": "07_generation_baselines",
    "summary": "08_final_summary",
}
for rel in SUBDIRS.values():
    (EVAL_ROOT / rel).mkdir(parents=True, exist_ok=True)
for rel in ["01_stage3_auc/recall_curves", "02_existing_stage4_checkpoint_eval/per_run", "02_existing_stage4_checkpoint_eval/recall_curves", "06_latent_diagnostics/plots"]:
    (EVAL_ROOT / rel).mkdir(parents=True, exist_ok=True)

def write_json(path: str | Path, payload: Any) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w") as f:
        json.dump(payload, f, indent=2, default=str)

def write_csv(path: str | Path, rows: list[dict[str, Any]]) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    rows = list(rows)
    if not rows:
        path.write_text("")
        return
    keys = sorted({k for row in rows for k in row})
    with path.open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=keys)
        writer.writeheader()
        writer.writerows(rows)

write_json(EVAL_ROOT / "00_metadata/evaluation_config.json", {
    "eval_root": str(EVAL_ROOT),
    "hf_dataset_repo": HF_DATASET_REPO,
    "unified_split_dir": str(UNIFIED_SPLIT_DIR),
    "text_embedding_cache": str(TEXT_EMBEDDING_CACHE),
    "primary_metric": "normalized_full_recall_curve_auc_over_k_div_N",
    "chance_auc": 0.5,
    "perfect_auc": 1.0,
    "modifies_existing_checkpoints": False,
})
print("Evaluation root:", EVAL_ROOT)


In [ ]:
# Explicit six-run registry. Defaults come from the completed run folder named below;
# override with environment variables only when your Drive layout differs.
DOMAIN_DIR = {"pubmed": "01_pubmed", "nilearn": "02_nilearn", "neurovault": "03_neurovault"}
SPECIALIZED_BRANCH = {
    "pubmed": "specialized_mixed_to_pubmed",
    "nilearn": "specialized_mixed_to_nilearn",
    "neurovault": "specialized_mixed_to_neurovault",
}
AE_REGISTRY_KEY = {
    ("pubmed", "baseline"): "mixed_stage1a",
    ("nilearn", "baseline"): "mixed_stage1a",
    ("neurovault", "baseline"): "mixed_stage1a",
    ("pubmed", "specialized"): "mixed_to_pubmed_stage1b",
    ("nilearn", "specialized"): "mixed_to_nilearn_stage1b",
    ("neurovault", "specialized"): "mixed_to_neurovault_stage1b",
}
DEFAULT_COMPLETED_STAGE3_RUN_ROOT = DRIVE_ROOT / "runs_stage2_stage3_stage4/stage2_stage3_stage4_20260625_210801"
COMPLETED_STAGE3_RUN_ROOT = Path(os.environ.get("NEUROVLM_COMPLETED_STAGE3_RUN_ROOT", str(DEFAULT_COMPLETED_STAGE3_RUN_ROOT))).expanduser()
LEGACY_STAGE4_RUN_ROOT = Path(os.environ.get("NEUROVLM_LEGACY_STAGE4_RUN_ROOT", str(COMPLETED_STAGE3_RUN_ROOT))).expanduser()

def find_downstream_manifest(run_root: Path) -> Path | None:
    candidates = [
        run_root / "00_upstream_stage1_selection/03_downstream_usage/stage2_stage3_stage4_input_manifest.json",
        run_root / "00_run_metadata/stage2_stage3_stage4_input_manifest.json",
        run_root / "00_metadata/stage2_stage3_stage4_input_manifest.json",
        run_root / "05_all_domain_comparison/stage2_stage3_stage4_input_manifest.json",
    ]
    pointer_candidates = [
        run_root / "00_run_metadata/stage2_stage3_stage4_input_manifest_pointer.json",
        run_root / "00_metadata/stage2_stage3_stage4_input_manifest_pointer.json",
        run_root / "05_all_domain_comparison/stage1_selection_integration_pointer.json",
    ]
    for pointer in pointer_candidates:
        if pointer.exists():
            payload = json.loads(pointer.read_text())
            for key in ["path", "downstream_manifest"]:
                if payload.get(key) and Path(payload[key]).exists():
                    candidates.insert(0, Path(payload[key]))
    for candidate in candidates:
        if candidate.exists():
            return candidate
    matches = sorted(run_root.glob("**/stage2_stage3_stage4_input_manifest.json")) if run_root.exists() else []
    return matches[0] if matches else None

DOWNSTREAM_MANIFEST_PATH = find_downstream_manifest(COMPLETED_STAGE3_RUN_ROOT)
DOWNSTREAM_MANIFEST = json.loads(DOWNSTREAM_MANIFEST_PATH.read_text()) if DOWNSTREAM_MANIFEST_PATH else {}
SELECTED_AE_CHECKPOINTS = DOWNSTREAM_MANIFEST.get("selected_ae_checkpoints", {})
if not DOWNSTREAM_MANIFEST_PATH:
    print("WARNING: downstream manifest not found under", COMPLETED_STAGE3_RUN_ROOT)
    print("AE checkpoint paths will use NEUROVLM_*_AE_CKPT overrides if provided; otherwise they are blank.")
else:
    print("Downstream manifest:", DOWNSTREAM_MANIFEST_PATH)

AE_ENV_BY_KEY = {
    "mixed_stage1a": "NEUROVLM_MIXED_STAGE1A_AE_CKPT",
    "mixed_to_pubmed_stage1b": "NEUROVLM_PUBMED_STAGE1B_AE_CKPT",
    "mixed_to_nilearn_stage1b": "NEUROVLM_NILEARN_STAGE1B_AE_CKPT",
    "mixed_to_neurovault_stage1b": "NEUROVLM_NEUROVAULT_STAGE1B_AE_CKPT",
}

def ae_checkpoint_for(domain: str, branch_kind: str) -> str:
    key = AE_REGISTRY_KEY[(domain, branch_kind)]
    override = os.environ.get(AE_ENV_BY_KEY[key], "").strip()
    if override:
        return str(Path(override).expanduser())
    entry = SELECTED_AE_CHECKPOINTS.get(key, {})
    return str(entry.get("path", ""))

def run_paths(domain: str, branch: str) -> dict[str, str]:
    base_stage3 = COMPLETED_STAGE3_RUN_ROOT / DOMAIN_DIR[domain] / branch
    base_stage4 = LEGACY_STAGE4_RUN_ROOT / DOMAIN_DIR[domain] / branch
    return {
        "stage3_checkpoint": str(base_stage3 / "stage3/checkpoints/best_ale_cnn.pt"),
        "stage3_run_dir": str(base_stage3 / "stage3"),
        "original_stage4_run_dir": str(base_stage4 / "stage4"),
    }

SOURCE_RUN_REGISTRY: list[dict[str, Any]] = []
for domain in ["pubmed", "nilearn", "neurovault"]:
    for branch_kind, branch in [("baseline", "baseline_mixed_stage1a"), ("specialized", SPECIALIZED_BRANCH[domain])]:
        ae_key = AE_REGISTRY_KEY[(domain, branch_kind)]
        ae_entry = SELECTED_AE_CHECKPOINTS.get(ae_key, {})
        row = {
            "domain": domain,
            "branch_kind": branch_kind,
            "branch": branch,
            "description": f"{'mixed Stage 1A baseline' if branch_kind == 'baseline' else 'mixed->' + domain + ' Stage 1B specialized'} on {domain}",
            "ae_registry_key": ae_key,
            "ae_checkpoint": ae_checkpoint_for(domain, branch_kind),
            "ae_checkpoint_name": ae_entry.get("checkpoint_name", ""),
            "train_manifest": str(UNIFIED_SPLIT_DIR / "train.jsonl"),
            "val_manifest": str(UNIFIED_SPLIT_DIR / "val.jsonl"),
            "test_manifest": str(UNIFIED_SPLIT_DIR / "test.jsonl"),
            "text_embedding_cache": str(TEXT_EMBEDDING_CACHE),
            "text_registry": str(TEXT_REGISTRY or ""),
            **run_paths(domain, branch),
        }
        row["original_stage4_checkpoints"] = [str(Path(row["original_stage4_run_dir"]) / "checkpoints" / name) for name in [
            "best_val_loss.pt", "best_generation_spatial_correlation.pt", "best_generation_top5_dice.pt",
            "best_val_spatial_corr.pt", "best_val_top5_dice.pt", "best_text_to_brain_decoder.pt", "last.pt", "last_text_to_brain_decoder.pt",
        ]]
        SOURCE_RUN_REGISTRY.append(row)

missing_required = []
for row in SOURCE_RUN_REGISTRY:
    for key in ["stage3_checkpoint", "original_stage4_run_dir", "train_manifest", "val_manifest", "test_manifest", "text_embedding_cache"]:
        if row[key] and not Path(row[key]).exists():
            missing_required.append({"run": f"{row['domain']}_{row['branch_kind']}", "field": key, "path": row[key]})
    if not row["ae_checkpoint"]:
        missing_required.append({"run": f"{row['domain']}_{row['branch_kind']}", "field": "ae_checkpoint", "path": "missing from downstream manifest/env"})
write_json(EVAL_ROOT / "00_metadata/source_run_registry.json", SOURCE_RUN_REGISTRY)
write_json(EVAL_ROOT / "00_metadata/source_run_registry_missing_paths.json", missing_required)
if missing_required:
    print("WARNING: some registered paths are missing. Evaluation cells will skip missing checkpoints or fail when required.")
    for item in missing_required[:20]:
        print(item)
pd.DataFrame(SOURCE_RUN_REGISTRY)


In [ ]:
# Normalized recall-curve AUC unit tests

def ranks_from_similarity(sim: torch.Tensor) -> torch.Tensor:
    order = sim.argsort(dim=1, descending=True)
    correct = torch.arange(sim.shape[0], device=sim.device)
    return order.eq(correct[:, None]).to(torch.int32).argmax(dim=1).float() + 1

def full_recall_curve_from_ranks(ranks: torch.Tensor, n: int | None = None) -> torch.Tensor:
    ranks = ranks.float()
    if n is None:
        n = int(ranks.numel())
    ks = torch.arange(1, n + 1, device=ranks.device).float()
    return (ranks[:, None] <= ks[None, :]).float().mean(dim=0)

def normalized_auc_from_ranks(ranks: torch.Tensor, n: int | None = None) -> float:
    return float(full_recall_curve_from_ranks(ranks, n).mean().item())

def bidirectional_auc_from_similarity(sim: torch.Tensor) -> dict[str, Any]:
    t2b_ranks = ranks_from_similarity(sim)
    b2t_ranks = ranks_from_similarity(sim.T)
    t2b_curve = full_recall_curve_from_ranks(t2b_ranks, sim.shape[1])
    b2t_curve = full_recall_curve_from_ranks(b2t_ranks, sim.shape[0])
    return {
        "text_to_brain_normalized_auc": float(t2b_curve.mean().item()),
        "brain_to_text_normalized_auc": float(b2t_curve.mean().item()),
        "mean_bidirectional_normalized_auc": float(((t2b_curve + b2t_curve) / 2).mean().item()),
        "text_to_brain_ranks": t2b_ranks.cpu(),
        "brain_to_text_ranks": b2t_ranks.cpu(),
        "text_to_brain_curve": t2b_curve.cpu(),
        "brain_to_text_curve": b2t_curve.cpu(),
    }

def test_normalized_auc():
    n = 16
    perfect = torch.eye(n)
    assert abs(bidirectional_auc_from_similarity(perfect)["mean_bidirectional_normalized_auc"] - 1.0) < 1e-8
    reversed_ranks = torch.arange(n, 0, -1)
    assert abs(normalized_auc_from_ranks(reversed_ranks, n) - 0.53125) < 1e-8
    manual = normalized_auc_from_ranks(torch.tensor([1, 2, 4, 4]), 4)
    assert abs(manual - ((1/4 + 2/4 + 2/4 + 4/4) / 4)) < 1e-8
    vals = []
    g = torch.Generator().manual_seed(0)
    for _ in range(200):
        vals.append(normalized_auc_from_ranks(torch.randperm(200, generator=g).float() + 1, 200))
    assert abs(float(np.mean(vals)) - 0.5) < 0.03
    return {"perfect": 1.0, "random_mean": float(np.mean(vals)), "manual": float(manual)}

auc_unit_test_results = test_normalized_auc()
write_json(EVAL_ROOT / "00_metadata/normalized_auc_unit_tests.json", auc_unit_test_results)
auc_unit_test_results


In [ ]:
# Shared loaders and model utilities

def read_jsonl(path: str | Path) -> list[dict[str, Any]]:
    with Path(path).open() as f:
        return [json.loads(line) for line in f if line.strip()]

def domain_from_source(value: str) -> str:
    source = str(value or "").lower()
    if source == "pubmed" or source.startswith("pubmed"):
        return "pubmed"
    if source == "nilearn" or source.startswith("nilearn"):
        return "nilearn"
    if source == "neurovault" or source.startswith("neurovault"):
        return "neurovault"
    return source

def primary_text(row: dict[str, Any]) -> dict[str, Any]:
    positives = row.get("positive_texts", []) or []
    if not positives:
        return {"text_id": "", "text": ""}
    first = positives[0]
    return {"text_id": str(first.get("text_id") or first.get("id") or first.get("text") or ""), "text": str(first.get("text", ""))}

def filter_rows(rows: list[dict[str, Any]], domain: str) -> list[dict[str, Any]]:
    out = [row for row in rows if domain_from_source(row.get("source", "")) == domain]
    bad = sorted({row.get("source", "") for row in out if domain_from_source(row.get("source", "")) != domain})
    if bad:
        raise RuntimeError(f"domain filter left non-domain rows for {domain}: {bad}")
    return out

def split_fingerprint(rows: list[dict[str, Any]]) -> str:
    pairs = sorted((str(row.get("map_id", "")), primary_text(row)["text_id"]) for row in rows)
    return hashlib.sha256(json.dumps(pairs, sort_keys=True, separators=(",", ":")).encode()).hexdigest()

def load_text_cache(path: str | Path) -> dict[str, torch.Tensor]:
    payload = torch.load(path, map_location="cpu", weights_only=False)
    if isinstance(payload, dict):
        for key in ["processed_embedding_by_text", "embedding_by_text", "text_to_embedding", "embeddings_by_text"]:
            if isinstance(payload.get(key), dict):
                payload = payload[key]
                break
        else:
            if isinstance(payload.get("records"), list):
                payload = {str(r.get("text", r.get("input_text", r.get("text_id", "")))): r["processed_embedding"] for r in payload["records"] if "processed_embedding" in r}
    return {str(k): torch.as_tensor(v, dtype=torch.float32) for k, v in payload.items() if torch.is_tensor(v) or isinstance(v, (list, tuple))}

def checkpoint_architecture(payload: dict[str, Any]) -> dict[str, Any]:
    cfg = payload.get("config", {}) if isinstance(payload, dict) else {}
    model_cfg = cfg.get("model", {}) if isinstance(cfg.get("model", {}), dict) else {}
    return {
        "latent_dim": int(model_cfg.get("latent_dim", payload.get("latent_dim", 384))),
        "base_channels": int(model_cfg.get("base_channels", cfg.get("base_channels", 64))),
        "num_blocks": int(model_cfg.get("num_blocks", cfg.get("num_blocks", 4))),
        "dropout": float(model_cfg.get("dropout", cfg.get("dropout", 0.1))),
        "norm": str(model_cfg.get("norm", cfg.get("norm", "group"))),
        "pooling": str(model_cfg.get("pooling", cfg.get("pooling", "max"))),
        "encoder_arch": str(model_cfg.get("encoder_arch", "plain")),
        "target_shape": tuple(payload.get("target_shape") or cfg.get("target_shape") or [36, 45, 38]),
    }

def load_stage3_models(path: str | Path, device: torch.device):
    payload = torch.load(path, map_location="cpu", weights_only=False)
    cfg = payload.get("config", {})
    encoder_arch = "plain" if cfg.get("model", "ale_3dcnn") == "ale_3dcnn" else "resnet"
    brain_encoder = build_brain_encoder(
        out_dim=int(cfg.get("out_dim", 384)), encoder_arch=encoder_arch,
        base_channels=int(cfg.get("base_channels", 64)), num_blocks=int(cfg.get("num_blocks", 4)),
        dropout=float(cfg.get("dropout", 0.1)), blocks_per_stage=int(cfg.get("blocks_per_stage", 2)),
        use_dilation=bool(cfg.get("use_dilation", False)), multi_scale=bool(cfg.get("multi_scale", False)),
        global_context=str(cfg.get("global_context", "none")),
    ).to(device)
    text_proj = build_text_projection("random", device=device)
    brain_encoder.load_state_dict(payload["brain_encoder"], strict=True)
    text_proj.load_state_dict(payload["text_proj"], strict=True)
    brain_encoder.eval(); text_proj.eval()
    for m in [brain_encoder, text_proj]:
        for p in m.parameters():
            p.requires_grad_(False)
    return brain_encoder, text_proj, payload

def load_autoencoder(path: str | Path, device: torch.device):
    payload = torch.load(path, map_location="cpu", weights_only=False)
    arch = checkpoint_architecture(payload)
    ae = build_cnn_autoencoder(
        arch["target_shape"], latent_dim=arch["latent_dim"], base_channels=arch["base_channels"],
        num_blocks=arch["num_blocks"], dropout=arch["dropout"], norm=arch["norm"], pooling=arch["pooling"],
        encoder_arch=arch["encoder_arch"],
    ).to(device)
    load_autoencoder_checkpoint(ae, path, strict=True)
    ae.eval()
    for p in ae.parameters():
        p.requires_grad_(False)
    return ae, arch, payload

def collate_primary(batch: list[dict[str, Any]], target_shape=(36,45,38)) -> dict[str, Any]:
    vols, texts, map_ids, text_ids, rows = [], [], [], [], []
    for item in batch:
        pos = item.get("positive_texts", []) or []
        if not pos:
            continue
        v = item["volume"].float()
        if tuple(v.shape[-3:]) != tuple(target_shape):
            v = F.interpolate(v.unsqueeze(0), size=target_shape, mode="trilinear", align_corners=False).squeeze(0)
        vols.append(v.clamp(0, 1)); texts.append(pos[0]["text"]); text_ids.append(pos[0].get("text_id", "")); map_ids.append(item["map_id"]); rows.append(item["metadata"])
    return {"volume": torch.stack(vols), "texts": texts, "text_ids": text_ids, "map_ids": map_ids, "rows": rows}


In [ ]:
# Stage 3 re-evaluation with normalized recall-curve AUC

def evaluate_stage3_run(reg: dict[str, Any], device: torch.device) -> tuple[dict[str, Any], pd.DataFrame]:
    rows = filter_rows(read_jsonl(reg["test_manifest"]), reg["domain"])
    ds = UnifiedMapTextDataset(reg["test_manifest"])
    ds.rows = rows
    text_cache = load_text_cache(reg["text_embedding_cache"])
    brain_encoder, text_proj, _ = load_stage3_models(reg["stage3_checkpoint"], device)
    loader = DataLoader(ds, batch_size=int(os.environ.get("NEUROVLM_EVAL_BATCH_SIZE", "128")), shuffle=False, collate_fn=lambda b: collate_primary(b))
    brain, text, map_ids, text_ids = [], [], [], []
    with torch.no_grad():
        for batch in loader:
            raw = torch.stack([text_cache[t] for t in batch["texts"]]).to(device)
            vol = batch["volume"].to(device)
            brain.append(brain_encoder(vol).cpu())
            text.append(text_proj(raw).cpu())
            map_ids.extend(batch["map_ids"]); text_ids.extend(batch["text_ids"])
    brain = torch.cat(brain); text = torch.cat(text)
    sim = F.normalize(text, dim=1) @ F.normalize(brain, dim=1).T
    auc = bidirectional_auc_from_similarity(sim)
    metrics = bidirectional_retrieval_metrics(text, brain, ks=(1,5,10,50))
    matched = sim.diag()
    shuffled = sim[torch.arange(sim.shape[0]), torch.randperm(sim.shape[0])] if sim.shape[0] > 1 else torch.tensor([float("nan")])
    out = {
        **{k: v for k, v in reg.items() if k in ["domain", "branch_kind", "branch", "description"]},
        "n_test": len(rows),
        "text_to_brain_normalized_auc": auc["text_to_brain_normalized_auc"],
        "brain_to_text_normalized_auc": auc["brain_to_text_normalized_auc"],
        "mean_bidirectional_normalized_auc": auc["mean_bidirectional_normalized_auc"],
        "matched_pair_cosine": float(matched.mean().item()),
        "shuffled_pair_cosine": float(shuffled.mean().item()),
        "mrr": metrics["mean_mrr"],
        "median_rank": metrics["mean_median_rank"],
        "recall@1": metrics["mean_recall@1"],
        "recall@5": metrics["mean_recall@5"],
        "recall@10": metrics["mean_recall@10"],
        "recall@50": metrics["mean_recall@50"],
        "stage3_checkpoint": reg["stage3_checkpoint"],
    }
    curve_df = pd.DataFrame({
        "k": np.arange(1, sim.shape[0] + 1),
        "k_over_N": np.arange(1, sim.shape[0] + 1) / sim.shape[0],
        "text_to_brain_recall": auc["text_to_brain_curve"].numpy(),
        "brain_to_text_recall": auc["brain_to_text_curve"].numpy(),
        "mean_recall": ((auc["text_to_brain_curve"] + auc["brain_to_text_curve"]) / 2).numpy(),
        "domain": reg["domain"], "branch_kind": reg["branch_kind"],
    })
    curve_path = EVAL_ROOT / f"01_stage3_auc/recall_curves/{reg['domain']}_{reg['branch_kind']}_stage3_recall_curve.csv"
    curve_df.to_csv(curve_path, index=False)
    return out, pd.DataFrame({"map_id": map_ids, "text_id": text_ids, "t2b_rank": auc["text_to_brain_ranks"].numpy(), "b2t_rank": auc["brain_to_text_ranks"].numpy()})

RUN_STAGE3_EVAL = os.environ.get("NEUROVLM_RUN_STAGE3_EVAL", "1") == "1"
stage3_rows, stage3_rank_tables = [], {}
if RUN_STAGE3_EVAL:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    for reg in SOURCE_RUN_REGISTRY:
        if not Path(reg["stage3_checkpoint"]).exists():
            print("Skipping missing Stage 3 checkpoint", reg["stage3_checkpoint"])
            continue
        row, ranks = evaluate_stage3_run(reg, device)
        stage3_rows.append(row)
        stage3_rank_tables[(reg["domain"], reg["branch_kind"])] = ranks
    write_csv(EVAL_ROOT / "01_stage3_auc/stage3_normalized_auc_all_runs.csv", stage3_rows)
    for domain in ["pubmed", "nilearn", "neurovault"]:
        write_csv(EVAL_ROOT / f"01_stage3_auc/{domain}_stage3_auc_baseline_vs_specialized.csv", [r for r in stage3_rows if r["domain"] == domain])
pd.DataFrame(stage3_rows)


In [ ]:
# Paired bootstrap CIs for specialized AUC - baseline AUC within domain

def auc_from_rank_subset(ranks: np.ndarray, n: int) -> float:
    return float(np.mean([(ranks <= k).mean() for k in range(1, n + 1)]))

def paired_bootstrap_auc_delta(domain: str, n_boot: int = 2000, seed: int = 0) -> dict[str, Any]:
    base = stage3_rank_tables.get((domain, "baseline"))
    spec = stage3_rank_tables.get((domain, "specialized"))
    if base is None or spec is None:
        return {"domain": domain, "status": "missing_ranks"}
    merged = base[["map_id", "text_id", "t2b_rank", "b2t_rank"]].merge(
        spec[["map_id", "text_id", "t2b_rank", "b2t_rank"]], on=["map_id", "text_id"], suffixes=("_baseline", "_specialized")
    )
    if merged.empty:
        return {"domain": domain, "status": "no_identical_test_examples"}
    rng = np.random.default_rng(seed)
    n = len(merged)
    def mean_auc(frame):
        return 0.5 * (auc_from_rank_subset(frame["t2b_rank"].to_numpy(), n) + auc_from_rank_subset(frame["b2t_rank"].to_numpy(), n))
    observed = mean_auc(merged.filter(regex="specialized$|map_id|text_id").rename(columns={"t2b_rank_specialized":"t2b_rank", "b2t_rank_specialized":"b2t_rank"})) - mean_auc(merged.filter(regex="baseline$|map_id|text_id").rename(columns={"t2b_rank_baseline":"t2b_rank", "b2t_rank_baseline":"b2t_rank"}))
    deltas = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        sample = merged.iloc[idx]
        spec_auc = 0.5 * (auc_from_rank_subset(sample["t2b_rank_specialized"].to_numpy(), n) + auc_from_rank_subset(sample["b2t_rank_specialized"].to_numpy(), n))
        base_auc = 0.5 * (auc_from_rank_subset(sample["t2b_rank_baseline"].to_numpy(), n) + auc_from_rank_subset(sample["b2t_rank_baseline"].to_numpy(), n))
        deltas.append(spec_auc - base_auc)
    return {"domain": domain, "n_paired_examples": n, "auc_delta_specialized_minus_baseline": observed, "ci_low": float(np.quantile(deltas, 0.025)), "ci_high": float(np.quantile(deltas, 0.975)), "status": "ok"}

bootstrap_rows = [paired_bootstrap_auc_delta(d) for d in ["pubmed", "nilearn", "neurovault"]]
write_json(EVAL_ROOT / "01_stage3_auc/paired_bootstrap_auc_delta.json", bootstrap_rows)
bootstrap_rows


In [ ]:
# Existing Stage 4 checkpoint de-duplication, architecture audit, and evaluation scaffolding
STAGE4_CANDIDATE_NAMES = [
    "best_val_loss.pt", "best_generation_spatial_correlation.pt", "best_generation_top5_dice.pt",
    "best_val_spatial_corr.pt", "best_val_top5_dice.pt", "best_text_to_brain_decoder.pt", "last.pt", "last_text_to_brain_decoder.pt",
]

def tensor_state_for_checksum(payload: dict[str, Any]) -> dict[str, torch.Tensor]:
    for key in ["generative_text_to_ae_latent", "text_projector", "text_to_brain_decoder", "state_dict", "model"]:
        if isinstance(payload.get(key), dict):
            return payload[key]
    return {k: v for k, v in payload.items() if torch.is_tensor(v)}

def state_checksum(path: str | Path) -> str:
    payload = torch.load(path, map_location="cpu", weights_only=False)
    state = tensor_state_for_checksum(payload)
    h = hashlib.sha256()
    for key in sorted(state):
        value = state[key].detach().cpu().contiguous()
        h.update(key.encode()); h.update(str(tuple(value.shape)).encode()); h.update(value.numpy().tobytes())
    return h.hexdigest()

def unique_stage4_checkpoints(reg: dict[str, Any]) -> list[dict[str, Any]]:
    seen = {}
    out = []
    ckpt_dir = Path(reg["original_stage4_run_dir"]) / "checkpoints"
    for name in STAGE4_CANDIDATE_NAMES:
        path = ckpt_dir / name
        if not path.exists():
            continue
        digest = state_checksum(path)
        if digest in seen:
            seen[digest]["aliases"].append(name)
            continue
        row = {"path": str(path), "primary_name": name, "aliases": [name], "state_checksum": digest}
        seen[digest] = row; out.append(row)
    return out

def architecture_audit_for_checkpoint(reg: dict[str, Any], ckpt: dict[str, Any]) -> dict[str, Any]:
    payload = torch.load(ckpt["path"], map_location="cpu", weights_only=False)
    cfg = payload.get("config", {}) if isinstance(payload, dict) else {}
    state = tensor_state_for_checksum(payload)
    first_weight = next((v for k, v in state.items() if torch.is_tensor(v) and v.ndim == 2), None)
    loaded_from_stage3 = cfg.get("text_projection_init") in {"pretrained_text_infonce", "text_infonce"} or bool(cfg.get("stage3_contrastive_checkpoint"))
    return {
        "domain": reg["domain"], "branch_kind": reg["branch_kind"], "checkpoint": ckpt["primary_name"], "aliases": ";".join(ckpt["aliases"]),
        "raw_input_embedding_dim": int(first_weight.shape[1]) if first_weight is not None else "unknown",
        "stage4_projector_architecture": json.dumps(cfg.get("generative_text_to_ae_latent") or cfg.get("text_to_brain_projection") or {}, sort_keys=True),
        "stage3_contrastive_checkpoint": cfg.get("stage3_contrastive_checkpoint", ""),
        "text_projection_init": cfg.get("text_projection_init", "unknown"),
        "improperly_reused_stage3_contrastive_projector": bool(loaded_from_stage3),
        "latent_target": cfg.get("loss_name", "legacy_or_unknown"),
        "decoder_trainability": "frozen_expected",
        "checkpoint_selection_metric": ckpt["primary_name"].removesuffix(".pt"),
    }

architecture_rows = []
transfer_report = {}
for reg in SOURCE_RUN_REGISTRY:
    unique = unique_stage4_checkpoints(reg) if Path(reg["original_stage4_run_dir"]).exists() else []
    transfer_report[f"{reg['domain']}_{reg['branch_kind']}"] = unique
    for ckpt in unique:
        architecture_rows.append(architecture_audit_for_checkpoint(reg, ckpt))
write_csv(EVAL_ROOT / "05_architecture_audit/existing_stage4_architecture_audit.csv", architecture_rows)
write_json(EVAL_ROOT / "05_architecture_audit/existing_stage4_projector_transfer_report.json", transfer_report)
pd.DataFrame(architecture_rows)


In [ ]:
# Stage 4 semantic/spatial checkpoint evaluation. This cell can be expensive; set RUN_STAGE4_EVAL=1 to execute.

def load_stage4_projector_from_checkpoint(path: str | Path, device: torch.device, latent_dim: int = 384):
    payload = torch.load(path, map_location="cpu", weights_only=False)
    cfg = payload.get("config", {})
    state = tensor_state_for_checksum(payload)
    if any(k.startswith("net.") for k in state):
        model = build_generative_text_to_ae_latent(device=device, in_dim=768, hidden_dim=512, latent_dim=latent_dim)
    else:
        proj_cfg = cfg.get("text_to_brain_projection", {})
        model = build_text_to_brain_projection("random", device=device, hidden_dim=int(proj_cfg.get("hidden_dim", 512)), depth=int(proj_cfg.get("depth", 2)), dropout=float(proj_cfg.get("dropout", 0.1)), out_dim=latent_dim)
    model.load_state_dict(state, strict=True)
    model.eval()
    for p in model.parameters(): p.requires_grad_(False)
    return model, payload

def evaluate_stage4_checkpoint(reg: dict[str, Any], ckpt: dict[str, Any], device: torch.device) -> dict[str, Any]:
    rows = filter_rows(read_jsonl(reg["test_manifest"]), reg["domain"])
    ds = UnifiedMapTextDataset(reg["test_manifest"]); ds.rows = rows
    ae, ae_arch, _ = load_autoencoder(reg["ae_checkpoint"], device)
    stage3_brain, stage3_text, _ = load_stage3_models(reg["stage3_checkpoint"], device)
    projector, payload = load_stage4_projector_from_checkpoint(ckpt["path"], device, latent_dim=ae_arch["latent_dim"])
    text_cache = load_text_cache(reg["text_embedding_cache"])
    loader = DataLoader(ds, batch_size=int(os.environ.get("NEUROVLM_EVAL_BATCH_SIZE", "64")), shuffle=False, collate_fn=lambda b: collate_primary(b, ae_arch["target_shape"]))
    gen_emb, text_emb, spatial_rows = [], [], []
    with torch.no_grad():
        for batch in loader:
            raw = torch.stack([text_cache[t] for t in batch["texts"]]).to(device)
            target = batch["volume"].to(device)
            pred_latent = projector(raw)
            pred = ae.decoder(pred_latent)
            gen_emb.append(stage3_brain(pred.float().clamp(0.0, 1.0)).cpu())
            text_emb.append(stage3_text(raw.float()).cpu())
            gm = generation_metrics(pred.detach().cpu(), target.detach().cpu(), include_voxel_auroc=False)
            spatial_rows.append(gm)
    gen_emb = torch.cat(gen_emb); text_emb = torch.cat(text_emb)
    sim = F.normalize(text_emb, dim=1) @ F.normalize(gen_emb, dim=1).T
    auc = bidirectional_auc_from_similarity(sim)
    matched = sim.diag(); shuffled = sim[torch.arange(sim.shape[0]), torch.randperm(sim.shape[0])] if sim.shape[0] > 1 else torch.tensor([float("nan")])
    spatial = {k: float(np.mean([r[k] for r in spatial_rows])) for k in spatial_rows[0]} if spatial_rows else {}
    curve_df = pd.DataFrame({"k": np.arange(1, sim.shape[0]+1), "k_over_N": np.arange(1, sim.shape[0]+1)/sim.shape[0], "text_to_generated_brain_recall": auc["text_to_brain_curve"].numpy(), "generated_brain_to_text_recall": auc["brain_to_text_curve"].numpy()})
    curve_stem = f"{reg['domain']}_{reg['branch_kind']}_{ckpt['primary_name'].removesuffix('.pt')}"
    curve_df.to_csv(EVAL_ROOT / f"02_existing_stage4_checkpoint_eval/recall_curves/{curve_stem}_generation_recall_curve.csv", index=False)
    write_json(EVAL_ROOT / f"02_existing_stage4_checkpoint_eval/recall_curves/{curve_stem}_generation_recall_curve.json", curve_df.to_dict(orient="list"))
    write_json(EVAL_ROOT / f"02_existing_stage4_checkpoint_eval/per_run/{curve_stem}_generation_similarity_matrix_metadata.json", {"n": sim.shape[0], "diagonal_is_true_pair": True, "matrix_orientation": "text rows x generated brain columns"})
    if sim.numel() <= int(os.environ.get("NEUROVLM_SAVE_SIM_MATRIX_MAX_ELEMENTS", "25000000")):
        torch.save(sim, EVAL_ROOT / f"02_existing_stage4_checkpoint_eval/per_run/{curve_stem}_similarity_matrix.pt")
    return {
        "domain": reg["domain"], "branch_kind": reg["branch_kind"], "checkpoint": ckpt["primary_name"], "aliases": ";".join(ckpt["aliases"]),
        "state_checksum": ckpt["state_checksum"], "n_test": len(rows),
        "text_to_generated_brain_normalized_auc": auc["text_to_brain_normalized_auc"],
        "generated_brain_to_text_normalized_auc": auc["brain_to_text_normalized_auc"],
        "mean_generation_normalized_recall_curve_auc": auc["mean_bidirectional_normalized_auc"],
        "matched_contrastive_cosine": float(matched.mean().item()), "shuffled_contrastive_cosine": float(shuffled.mean().item()),
        **spatial,
    }

stage4_rows = []
if os.environ.get("NEUROVLM_RUN_STAGE4_EVAL", "0") == "1":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    for reg in SOURCE_RUN_REGISTRY:
        for ckpt in unique_stage4_checkpoints(reg):
            stage4_rows.append(evaluate_stage4_checkpoint(reg, ckpt, device))
    write_csv(EVAL_ROOT / "02_existing_stage4_checkpoint_eval/stage4_checkpoint_comparison_all_runs.csv", stage4_rows)
    for domain in ["pubmed", "nilearn", "neurovault"]:
        domain_rows = [r for r in stage4_rows if r["domain"] == domain]
        write_csv(EVAL_ROOT / f"02_existing_stage4_checkpoint_eval/{domain}_stage4_checkpoint_comparison.csv", domain_rows)
    selected = {}
    for (domain, branch), frame in pd.DataFrame(stage4_rows).groupby(["domain", "branch_kind"]):
        best = frame.sort_values("mean_generation_normalized_recall_curve_auc", ascending=False).iloc[0].to_dict()
        selected[f"{domain}_{branch}"] = best
    write_json(EVAL_ROOT / "02_existing_stage4_checkpoint_eval/selected_existing_stage4_checkpoints.json", selected)
else:
    print("Set NEUROVLM_RUN_STAGE4_EVAL=1 to evaluate all unique legacy Stage 4 checkpoints.")


In [ ]:
# Pairing audit: single-positive policy, duplicate text groups, split fingerprints
pairing_rows, map_primary_rows = [], []
pairing_summary = {}
for split_name in ["train", "val", "test"]:
    path = UNIFIED_SPLIT_DIR / f"{split_name}.jsonl"
    if not path.exists():
        continue
    rows = read_jsonl(path)
    for domain in ["pubmed", "nilearn", "neurovault"]:
        drows = filter_rows(rows, domain)
        text_counts_per_map = [len(r.get("positive_texts", []) or []) for r in drows]
        primary_ids = [primary_text(r)["text_id"] for r in drows]
        maps_per_text = Counter(primary_ids)
        duplicate_titles = Counter((primary_text(r)["text"] or "").strip().lower() for r in drows)
        duplicate_title_groups = sum(1 for _, c in duplicate_titles.items() if c > 1)
        row = {
            "split": split_name, "domain": domain, "n_maps": len(drows), "unique_text_ids": len(set(primary_ids)),
            "texts_per_map_min": min(text_counts_per_map) if text_counts_per_map else 0,
            "texts_per_map_max": max(text_counts_per_map) if text_counts_per_map else 0,
            "texts_per_map_mean": float(np.mean(text_counts_per_map)) if text_counts_per_map else 0,
            "maps_per_text_max": max(maps_per_text.values()) if maps_per_text else 0,
            "duplicated_title_abstract_groups": duplicate_title_groups,
            "source_counts": json.dumps(Counter(str(r.get("source", "")) for r in drows), sort_keys=True),
            "split_fingerprint": split_fingerprint(drows),
            "completed_experiment_policy": "single_positive_primary_text_per_map" if all(c >= 1 for c in text_counts_per_map) else "missing_positive_texts",
        }
        pairing_rows.append(row)
        for r in drows:
            pt = primary_text(r)
            map_primary_rows.append({"split": split_name, "domain": domain, "map_id": r.get("map_id", ""), "primary_text_id": pt["text_id"], "primary_text": pt["text"], "source": r.get("source", ""), "publication_id": r.get("publication_id", r.get("pmid", ""))})
        pairing_summary[f"{split_name}_{domain}"] = row
write_csv(EVAL_ROOT / "04_pairing_audit/text_pairing_audit.csv", pairing_rows)
write_json(EVAL_ROOT / "04_pairing_audit/text_pairing_audit.json", pairing_summary)
write_csv(EVAL_ROOT / "04_pairing_audit/map_to_primary_text_manifest.csv", map_primary_rows)
write_json(EVAL_ROOT / "00_metadata/split_fingerprints.json", {f"{r['split']}_{r['domain']}": r["split_fingerprint"] for r in pairing_rows})
pd.DataFrame(pairing_rows)


In [ ]:
# Text embedding cache audit

def audit_text_cache(cache_path: str | Path, split_rows: list[dict[str, Any]] | None = None, sample_size: int = 2048) -> tuple[list[dict[str, Any]], dict[str, Any]]:
    cache = load_text_cache(cache_path)
    keys = list(cache)
    mat = torch.stack([cache[k].float() for k in keys]) if keys else torch.empty(0, 0)
    norms = mat.norm(dim=1) if mat.numel() else torch.tensor([])
    rng = np.random.default_rng(0)
    subset_idx = rng.choice(len(keys), size=min(sample_size, len(keys)), replace=False) if keys else []
    sub = F.normalize(mat[subset_idx], dim=1, eps=1e-8) if len(subset_idx) else torch.empty(0,0)
    pairwise = (sub @ sub.T) if len(subset_idx) else torch.empty(0,0)
    if pairwise.numel() and pairwise.shape[0] > 1:
        mean_pairwise = float(pairwise[~torch.eye(pairwise.shape[0], dtype=torch.bool)].mean().item())
    else:
        mean_pairwise = float("nan")
    duplicate_fraction = 0.0
    if keys:
        hashes = [hashlib.sha256(mat[i].numpy().tobytes()).hexdigest() for i in range(mat.shape[0])]
        duplicate_fraction = 1.0 - len(set(hashes)) / len(hashes)
    alignment_errors = []
    if split_rows:
        for row in split_rows:
            text = primary_text(row)["text"]
            if text not in cache:
                alignment_errors.append({"map_id": row.get("map_id", ""), "text_id": primary_text(row)["text_id"]})
    summary = {
        "cache_path": str(cache_path), "n_vectors": len(keys), "dimension": int(mat.shape[1]) if mat.ndim == 2 and mat.shape[0] else None,
        "norm_mean": float(norms.mean().item()) if len(norms) else float("nan"),
        "norm_std": float(norms.std().item()) if len(norms) else float("nan"),
        "norm_min": float(norms.min().item()) if len(norms) else float("nan"),
        "norm_max": float(norms.max().item()) if len(norms) else float("nan"),
        "norm_median": float(norms.median().item()) if len(norms) else float("nan"),
        "fraction_norm_within_1e_4_of_1": float((norms.sub(1).abs() <= 1e-4).float().mean().item()) if len(norms) else float("nan"),
        "fraction_norm_within_1e_3_of_1": float((norms.sub(1).abs() <= 1e-3).float().mean().item()) if len(norms) else float("nan"),
        "per_dimension_mean_mean": float(mat.mean(dim=0).mean().item()) if mat.numel() else float("nan"),
        "mean_pairwise_cosine_subset": mean_pairwise,
        "fraction_duplicate_vectors": duplicate_fraction,
        "nan_count": int(torch.isnan(mat).sum().item()) if mat.numel() else 0,
        "inf_count": int(torch.isinf(mat).sum().item()) if mat.numel() else 0,
        "text_id_alignment_error_count": len(alignment_errors),
        "alignment_error_examples": alignment_errors[:20],
    }
    return [summary], summary

all_split_rows = []
for name in ["train", "val", "test"]:
    p = UNIFIED_SPLIT_DIR / f"{name}.jsonl"
    if p.exists(): all_split_rows.extend(read_jsonl(p))
cache_rows, cache_summary = audit_text_cache(TEXT_EMBEDDING_CACHE, all_split_rows)
write_csv(EVAL_ROOT / "03_text_cache_audit/text_embedding_norm_audit.csv", cache_rows)
write_json(EVAL_ROOT / "03_text_cache_audit/text_embedding_cache_audit.json", cache_summary)
cache_summary


In [ ]:
# Reproduce sample cached embeddings and decide whether Stage 4 re-encoding is required.
# If exact encoder metadata is unavailable, the cache is marked non-reproducible.
REPRODUCE_EMBEDDINGS = os.environ.get("NEUROVLM_REPRODUCE_TEXT_EMBEDDINGS", "0") == "1"
reproduction_rows = []
reproduction_summary = {
    "status": "not_run",
    "encoder_name": "allenai/specter2_aug2023refresh_candidate",
    "encoder_revision": os.environ.get("NEUROVLM_SPECTER2_REVISION", "unknown"),
    "tokenizer_revision": os.environ.get("NEUROVLM_SPECTER2_REVISION", "unknown"),
    "pooling_method": "Specter wrapper default",
    "max_token_length": "unknown_unless_reproduced",
    "truncation_policy": "unknown_unless_reproduced",
    "title_abstract_separator": "source JSONL positive_texts[0].text",
    "embedding_dimension": cache_summary.get("dimension"),
}
if REPRODUCE_EMBEDDINGS:
    from atlas_free_cnn.training.model_wrappers import encode_texts_specter
    cache = load_text_cache(TEXT_EMBEDDING_CACHE)
    texts = [primary_text(r)["text"] for r in all_split_rows if primary_text(r)["text"] in cache]
    rng = random.Random(0); rng.shuffle(texts); texts = texts[: int(os.environ.get("NEUROVLM_REPRO_SAMPLE_N", "24"))]
    device = "cuda" if torch.cuda.is_available() else "cpu"
    raw = encode_texts_specter(texts + [""], device=device, batch_size=8)
    empty = raw[-1]
    candidates = {
        "raw_embedding": raw[:-1],
        "unit_normalized_raw_embedding": F.normalize(raw[:-1], dim=1, eps=1e-8),
        "raw_minus_empty": raw[:-1] - empty,
        "unit_normalized_raw_minus_empty": F.normalize(raw[:-1] - empty, dim=1, eps=1e-8),
    }
    for i, text in enumerate(texts):
        cached = cache[text].float()
        for name, value in candidates.items():
            diff = value[i].cpu() - cached
            reproduction_rows.append({"text_index": i, "candidate_transform": name, "cosine": float(F.cosine_similarity(value[i].cpu(), cached, dim=0, eps=1e-8).item()), "max_abs_diff": float(diff.abs().max().item()), "mse": float(diff.pow(2).mean().item())})
    rep = pd.DataFrame(reproduction_rows)
    best = rep.groupby("candidate_transform")["mse"].mean().sort_values().index[0]
    reproduction_summary.update({"status": "run", "best_transform": best, "empty_string_embedding_checksum": hashlib.sha256(empty.cpu().numpy().tobytes()).hexdigest()})
else:
    reproduction_summary.update({"status": "non_reproducible_without_explicit_run", "reason": "Set NEUROVLM_REPRODUCE_TEXT_EMBEDDINGS=1 in an environment with the exact SPECTER2 dependencies to numerically reproduce the cache."})

write_csv(EVAL_ROOT / "03_text_cache_audit/text_embedding_reproduction_audit.csv", reproduction_rows)
write_json(EVAL_ROOT / "03_text_cache_audit/text_embedding_reproduction_audit.json", reproduction_summary)
REENCODE_STAGE4_TEXT = (
    reproduction_summary["status"] != "run" or
    reproduction_summary.get("best_transform") != "unit_normalized_raw_minus_empty" or
    cache_summary.get("dimension") != 768 or
    cache_summary.get("nan_count", 0) > 0 or cache_summary.get("inf_count", 0) > 0 or
    cache_summary.get("text_id_alignment_error_count", 0) > 0 or
    cache_summary.get("fraction_duplicate_vectors", 0) > 0
)
write_json(EVAL_ROOT / "03_text_cache_audit/reencoding_decision.json", {"REENCODE_STAGE4_TEXT": bool(REENCODE_STAGE4_TEXT), "reason": "Prefer a versioned SPECTER2 generation cache for corrected Stage 4; old contrastive cache is left untouched."})
{"REENCODE_STAGE4_TEXT": REENCODE_STAGE4_TEXT, **reproduction_summary}


In [ ]:
# Create a new SPECTER2 generation cache when required. This writes a new cache only; it never overwrites Stage 3's cache.
CREATE_STAGE4_CACHE = os.environ.get("NEUROVLM_CREATE_STAGE4_SPECTER2_CACHE", "0") == "1"
if CREATE_STAGE4_CACHE:
    from atlas_free_cnn.training.model_wrappers import encode_texts_specter
    unique = {}
    for row in all_split_rows:
        pt = primary_text(row)
        if pt["text_id"] and pt["text"]:
            unique.setdefault(pt["text_id"], pt["text"])
    text_ids = sorted(unique)
    texts = [unique[t] for t in text_ids]
    device = "cuda" if torch.cuda.is_available() else "cpu"
    raw = encode_texts_specter(texts + [""], device=device, batch_size=int(os.environ.get("NEUROVLM_SPECTER_BATCH", "16")))
    empty = raw[-1]
    processed = F.normalize(raw[:-1] - empty, dim=1, eps=1e-8)
    model_revision = os.environ.get("NEUROVLM_SPECTER2_REVISION", "unknown_revision").replace("/", "_")
    cache_path = EVAL_ROOT / f"03_text_cache_audit/specter2_stage4_demeaned_unitnorm_{model_revision}.pt"
    payload = {
        "processed_embedding_by_text": {text: processed[i].cpu() for i, text in enumerate(texts)},
        "processed_embedding_by_text_id": {tid: processed[i].cpu() for i, tid in enumerate(text_ids)},
        "raw_embedding_by_text_id": {tid: raw[i].cpu() for i, tid in enumerate(text_ids)},
        "records": [
            {"text_id": tid, "text": texts[i], "raw_norm": float(raw[i].norm().item()), "processed_norm": float(processed[i].norm().item()), "text_checksum": hashlib.sha256(texts[i].encode()).hexdigest(), "embedding_checksum": hashlib.sha256(processed[i].cpu().numpy().tobytes()).hexdigest()}
            for i, tid in enumerate(text_ids)
        ],
        "metadata": {"encoder_name": "allenai/specter2_aug2023refresh", "encoder_revision": model_revision, "pooling_method": "Specter wrapper default", "empty_string_embedding_checksum": hashlib.sha256(empty.cpu().numpy().tobytes()).hexdigest(), "embedding_dimension": 768, "processing": "raw_minus_empty_then_unit_normalize"},
    }
    torch.save(payload, cache_path)
    write_json(EVAL_ROOT / "03_text_cache_audit/stage4_generation_cache_metadata.json", {"cache_path": str(cache_path), **payload["metadata"]})
    print("Wrote", cache_path)
else:
    print("Set NEUROVLM_CREATE_STAGE4_SPECTER2_CACHE=1 to create the versioned Stage 4 SPECTER2 generation cache when required.")


In [ ]:
# Latent-distribution diagnostics and non-neural/simple generation baselines.
# These sections are expensive, but unlike the earlier scaffold they actually run when enabled.
RUN_LATENT_DIAGNOSTICS = os.environ.get("NEUROVLM_RUN_LATENT_DIAGNOSTICS", "0") == "1"
RUN_GENERATION_BASELINES = os.environ.get("NEUROVLM_RUN_GENERATION_BASELINES", "0") == "1"
LATENT_BATCH_SIZE = int(os.environ.get("NEUROVLM_LATENT_DIAGNOSTIC_BATCH_SIZE", os.environ.get("NEUROVLM_EVAL_BATCH_SIZE", "64")))
BASELINE_BATCH_SIZE = int(os.environ.get("NEUROVLM_BASELINE_BATCH_SIZE", os.environ.get("NEUROVLM_EVAL_BATCH_SIZE", "64")))
RIDGE_ALPHA = float(os.environ.get("NEUROVLM_BASELINE_RIDGE_ALPHA", "1.0"))
MLP_EPOCHS = int(os.environ.get("NEUROVLM_BASELINE_MLP_EPOCHS", "25"))
MLP_LR = float(os.environ.get("NEUROVLM_BASELINE_MLP_LR", "5e-5"))
MAX_BASELINE_TRAIN_EXAMPLES = int(os.environ.get("NEUROVLM_BASELINE_MAX_TRAIN_EXAMPLES", "0")) or None

latent_rows: list[dict[str, Any]] = []
latent_summary_rows: list[dict[str, Any]] = []
baseline_rows: list[dict[str, Any]] = []
latent_errors: list[dict[str, Any]] = []
baseline_errors: list[dict[str, Any]] = []


def selected_stage4_checkpoint_for_reg(reg: dict[str, Any]) -> dict[str, Any] | None:
    unique = unique_stage4_checkpoints(reg) if Path(reg["original_stage4_run_dir"]).exists() else []
    if not unique:
        return None
    if "stage4_rows" in globals() and stage4_rows:
        frame = pd.DataFrame(stage4_rows)
        mask = (frame["domain"] == reg["domain"]) & (frame["branch_kind"] == reg["branch_kind"])
        frame = frame[mask]
        if not frame.empty and "mean_generation_normalized_recall_curve_auc" in frame:
            best = frame.sort_values("mean_generation_normalized_recall_curve_auc", ascending=False).iloc[0]
            for ckpt in unique:
                if ckpt["state_checksum"] == best.get("state_checksum"):
                    return ckpt
    priority = [
        "best_val_generation_normalized_auc.pt",
        "best_generation_spatial_correlation.pt",
        "best_generation_top5_dice.pt",
        "best_val_spatial_corr.pt",
        "best_val_top5_dice.pt",
        "best_val_loss.pt",
        "last.pt",
    ]
    for name in priority:
        for ckpt in unique:
            if ckpt["primary_name"] == name or name in ckpt.get("aliases", []):
                return ckpt
    return unique[0]


def make_domain_dataset(jsonl_path: str | Path, domain: str, *, load_volumes: bool = True) -> UnifiedMapTextDataset:
    ds = UnifiedMapTextDataset(jsonl_path, load_volumes=load_volumes)
    ds.rows = filter_rows(ds.rows, domain)
    return ds


def stack_text_cache(text_cache: dict[str, torch.Tensor], texts: list[str]) -> torch.Tensor:
    missing = [t for t in texts if t not in text_cache]
    if missing:
        raise KeyError(f"{len(missing)} texts missing from cache; first={missing[0][:160]}")
    return torch.stack([text_cache[t].float() for t in texts])


@torch.no_grad()
def collect_latent_eval_arrays(reg: dict[str, Any], ckpt: dict[str, Any], device: torch.device) -> dict[str, Any]:
    ae, ae_arch, _ = load_autoencoder(reg["ae_checkpoint"], device)
    projector, _ = load_stage4_projector_from_checkpoint(ckpt["path"], device, latent_dim=ae_arch["latent_dim"])
    text_cache = load_text_cache(reg["text_embedding_cache"])
    train_ds = make_domain_dataset(reg["train_manifest"], reg["domain"])
    test_ds = make_domain_dataset(reg["test_manifest"], reg["domain"])

    train_latents, train_norms = [], []
    train_loader = DataLoader(train_ds, batch_size=LATENT_BATCH_SIZE, shuffle=False, collate_fn=lambda b: collate_primary(b, ae_arch["target_shape"]))
    for batch in train_loader:
        vol = batch["volume"].to(device)
        z = ae.encoder(vol).float().cpu()
        train_latents.append(z)
        train_norms.append(z.norm(dim=1))
    train_latents = torch.cat(train_latents, dim=0)
    train_norms = torch.cat(train_norms, dim=0)

    true_latents, pred_latents, map_ids, text_ids = [], [], [], []
    decoded_true_metric_rows, decoded_pred_metric_rows = [], []
    test_loader = DataLoader(test_ds, batch_size=LATENT_BATCH_SIZE, shuffle=False, collate_fn=lambda b: collate_primary(b, ae_arch["target_shape"]))
    for batch in test_loader:
        raw = stack_text_cache(text_cache, batch["texts"]).to(device)
        target = batch["volume"].to(device)
        true_z = ae.encoder(target).float()
        pred_z = projector(raw).float()
        decoded_true = ae.decoder(true_z)
        decoded_pred = ae.decoder(pred_z)
        true_latents.append(true_z.cpu())
        pred_latents.append(pred_z.cpu())
        map_ids.extend(batch["map_ids"])
        text_ids.extend(batch["text_ids"])
        decoded_true_metric_rows.append(generation_metrics(decoded_true.detach().cpu(), target.detach().cpu(), include_voxel_auroc=False))
        decoded_pred_metric_rows.append(generation_metrics(decoded_pred.detach().cpu(), target.detach().cpu(), include_voxel_auroc=False))
    return {
        "ae": ae,
        "ae_arch": ae_arch,
        "projector": projector,
        "text_cache": text_cache,
        "train_ds": train_ds,
        "test_ds": test_ds,
        "train_latents": train_latents,
        "train_norms": train_norms,
        "true_latents": torch.cat(true_latents, dim=0),
        "pred_latents": torch.cat(pred_latents, dim=0),
        "map_ids": map_ids,
        "text_ids": text_ids,
        "decoded_true_metrics": decoded_true_metric_rows,
        "decoded_pred_metrics": decoded_pred_metric_rows,
    }


def nearest_real_latent_distances(pred: torch.Tensor, train: torch.Tensor, chunk: int = 512) -> torch.Tensor:
    out = []
    for start in range(0, pred.shape[0], chunk):
        dist = torch.cdist(pred[start:start + chunk].float(), train.float())
        out.append(dist.min(dim=1).values.cpu())
    return torch.cat(out, dim=0)


def pca_two_components(true_z: torch.Tensor, pred_z: torch.Tensor) -> tuple[np.ndarray, np.ndarray]:
    x = torch.cat([true_z.float(), pred_z.float()], dim=0)
    x = x - x.mean(dim=0, keepdim=True)
    try:
        _, _, v = torch.pca_lowrank(x, q=2)
        coords = x @ v[:, :2]
    except Exception:
        u, s, v = torch.linalg.svd(x, full_matrices=False)
        coords = x @ v[:2].T
    n = true_z.shape[0]
    return coords[:n].numpy(), coords[n:].numpy()


def aggregate_metric_rows(rows: list[dict[str, float]], prefix: str) -> dict[str, float]:
    if not rows:
        return {}
    keys = rows[0].keys()
    return {f"{prefix}_{k}": float(np.nanmean([row[k] for row in rows])) for k in keys}


def run_latent_diagnostics_for_reg(reg: dict[str, Any], device: torch.device) -> tuple[list[dict[str, Any]], dict[str, Any]]:
    ckpt = selected_stage4_checkpoint_for_reg(reg)
    if ckpt is None:
        raise FileNotFoundError(f"No legacy Stage 4 checkpoint found for {reg['domain']} {reg['branch_kind']}")
    arrays = collect_latent_eval_arrays(reg, ckpt, device)
    true_z = arrays["true_latents"]
    pred_z = arrays["pred_latents"]
    train_z = arrays["train_latents"]
    train_norms = arrays["train_norms"]
    latent_mse = (pred_z - true_z).pow(2).mean(dim=1)
    latent_cos = F.cosine_similarity(pred_z, true_z, dim=1, eps=1e-8)
    euclidean = (pred_z - true_z).norm(dim=1)
    true_norm = true_z.norm(dim=1)
    pred_norm = pred_z.norm(dim=1)
    nn_dist = nearest_real_latent_distances(pred_z, train_z, chunk=int(os.environ.get("NEUROVLM_NEAREST_LATENT_CHUNK", "512")))
    train_min = float(train_norms.min().item())
    train_max = float(train_norms.max().item())
    outside = ((pred_norm < train_min) | (pred_norm > train_max)).float()
    true_pca, pred_pca = pca_two_components(true_z, pred_z)

    per_rows = []
    for i, map_id in enumerate(arrays["map_ids"]):
        per_rows.append({
            "domain": reg["domain"],
            "branch_kind": reg["branch_kind"],
            "checkpoint": ckpt["primary_name"],
            "state_checksum": ckpt["state_checksum"],
            "map_id": map_id,
            "text_id": arrays["text_ids"][i],
            "latent_mse": float(latent_mse[i].item()),
            "latent_cosine": float(latent_cos[i].item()),
            "latent_euclidean_distance": float(euclidean[i].item()),
            "true_latent_norm": float(true_norm[i].item()),
            "pred_latent_norm": float(pred_norm[i].item()),
            "nearest_real_latent_distance": float(nn_dist[i].item()),
            "pred_norm_outside_training_range": bool(outside[i].item()),
            "true_latent_pca1": float(true_pca[i, 0]),
            "true_latent_pca2": float(true_pca[i, 1]),
            "pred_latent_pca1": float(pred_pca[i, 0]),
            "pred_latent_pca2": float(pred_pca[i, 1]),
        })

    true_mean = true_z.mean(dim=0)
    pred_mean = pred_z.mean(dim=0)
    true_std = true_z.std(dim=0)
    pred_std = pred_z.std(dim=0)
    true_centered = true_z - true_mean
    pred_centered = pred_z - pred_mean
    cov_true = true_centered.T @ true_centered / max(1, true_z.shape[0] - 1)
    cov_pred = pred_centered.T @ pred_centered / max(1, pred_z.shape[0] - 1)
    cov_diff = torch.linalg.matrix_norm(cov_true - cov_pred).item()
    cov_true_norm = torch.linalg.matrix_norm(cov_true).item()
    summary = {
        "domain": reg["domain"],
        "branch_kind": reg["branch_kind"],
        "checkpoint": ckpt["primary_name"],
        "state_checksum": ckpt["state_checksum"],
        "n_test": int(true_z.shape[0]),
        "n_train_latents": int(train_z.shape[0]),
        "latent_mse_mean": float(latent_mse.mean().item()),
        "latent_mse_median": float(latent_mse.median().item()),
        "latent_cosine_mean": float(latent_cos.mean().item()),
        "latent_euclidean_distance_mean": float(euclidean.mean().item()),
        "true_latent_norm_mean": float(true_norm.mean().item()),
        "pred_latent_norm_mean": float(pred_norm.mean().item()),
        "train_latent_norm_min": train_min,
        "train_latent_norm_max": train_max,
        "percent_pred_norm_outside_training_range": float(outside.mean().item() * 100.0),
        "nearest_real_latent_distance_mean": float(nn_dist.mean().item()),
        "per_dimension_mean_abs_difference": float((true_mean - pred_mean).abs().mean().item()),
        "per_dimension_std_abs_difference": float((true_std - pred_std).abs().mean().item()),
        "covariance_frobenius_difference": float(cov_diff),
        "covariance_relative_frobenius_difference": float(cov_diff / max(cov_true_norm, 1e-8)),
        **aggregate_metric_rows(arrays["decoded_true_metrics"], "decoded_true_latent"),
        **aggregate_metric_rows(arrays["decoded_pred_metrics"], "decoded_pred_latent"),
    }

    if plt is not None:
        plot_dir = EVAL_ROOT / "06_latent_diagnostics/plots"
        plot_dir.mkdir(parents=True, exist_ok=True)
        stem = f"{reg['domain']}_{reg['branch_kind']}"
        fig, ax = plt.subplots(figsize=(6, 5))
        ax.scatter(true_pca[:, 0], true_pca[:, 1], s=8, alpha=0.35, label="true AE latent")
        ax.scatter(pred_pca[:, 0], pred_pca[:, 1], s=8, alpha=0.35, label="predicted text latent")
        ax.set_title(f"Latent PCA: {stem}")
        ax.legend()
        fig.tight_layout()
        fig.savefig(plot_dir / f"latent_pca_{stem}.png", dpi=150)
        plt.close(fig)
        fig, ax = plt.subplots(figsize=(6, 4))
        ax.hist(true_norm.numpy(), bins=40, alpha=0.5, label="true")
        ax.hist(pred_norm.numpy(), bins=40, alpha=0.5, label="pred")
        ax.axvline(train_min, color="black", linestyle="--", linewidth=1)
        ax.axvline(train_max, color="black", linestyle="--", linewidth=1)
        ax.set_title(f"Latent Norms: {stem}")
        ax.legend()
        fig.tight_layout()
        fig.savefig(plot_dir / f"latent_norm_distribution_{stem}.png", dpi=150)
        plt.close(fig)
    return per_rows, summary


@torch.no_grad()
def build_reference_cache(reg: dict[str, Any], ae, ae_arch, stage3_text, text_cache, device: torch.device) -> dict[str, Any]:
    train_ds = make_domain_dataset(reg["train_manifest"], reg["domain"])
    if MAX_BASELINE_TRAIN_EXAMPLES is not None and len(train_ds.rows) > MAX_BASELINE_TRAIN_EXAMPLES:
        train_ds.rows = train_ds.rows[:MAX_BASELINE_TRAIN_EXAMPLES]
    loader = DataLoader(train_ds, batch_size=BASELINE_BATCH_SIZE, shuffle=False, collate_fn=lambda b: collate_primary(b, ae_arch["target_shape"]))
    raw_texts, stage3_texts, latents = [], [], []
    volume_sum = None
    n_vox = 0
    for batch in loader:
        raw = stack_text_cache(text_cache, batch["texts"]).to(device)
        vol = batch["volume"].to(device)
        z = ae.encoder(vol).float().cpu()
        txt = stage3_text(raw.float()).cpu()
        raw_texts.append(raw.cpu())
        stage3_texts.append(txt)
        latents.append(z)
        batch_sum = vol.detach().cpu().sum(dim=0)
        volume_sum = batch_sum if volume_sum is None else volume_sum + batch_sum
        n_vox += vol.shape[0]
    return {
        "train_ds": train_ds,
        "train_raw_text": torch.cat(raw_texts, dim=0),
        "train_stage3_text": torch.cat(stage3_texts, dim=0),
        "train_latents": torch.cat(latents, dim=0),
        "mean_map": (volume_sum / max(1, n_vox)).float(),
    }


def load_train_volumes_by_indices(train_ds: UnifiedMapTextDataset, indices: torch.Tensor, target_shape: tuple[int, int, int]) -> torch.Tensor:
    vols = []
    for idx in indices.cpu().tolist():
        item = train_ds[int(idx)]
        v = item["volume"].float()
        if tuple(v.shape[-3:]) != tuple(target_shape):
            v = F.interpolate(v.unsqueeze(0), size=target_shape, mode="trilinear", align_corners=False).squeeze(0)
        vols.append(v.clamp(0, 1))
    return torch.stack(vols)


def fit_ridge_closed_form(x: torch.Tensor, y: torch.Tensor, alpha: float) -> torch.Tensor:
    x = x.float()
    y = y.float()
    ones = torch.ones((x.shape[0], 1), dtype=x.dtype)
    xb = torch.cat([x, ones], dim=1)
    eye = torch.eye(xb.shape[1], dtype=x.dtype)
    eye[-1, -1] = 0.0
    return torch.linalg.solve(xb.T @ xb + float(alpha) * eye, xb.T @ y)


def predict_ridge(x: torch.Tensor, w: torch.Tensor) -> torch.Tensor:
    ones = torch.ones((x.shape[0], 1), dtype=x.dtype)
    return torch.cat([x.float(), ones], dim=1) @ w


def train_diagnostic_mlp(x: torch.Tensor, y: torch.Tensor, device: torch.device) -> nn.Module:
    model = build_generative_text_to_ae_latent(device=device, in_dim=x.shape[1], hidden_dim=512, latent_dim=y.shape[1])
    opt = torch.optim.AdamW(model.parameters(), lr=MLP_LR, weight_decay=1e-4)
    ds = torch.utils.data.TensorDataset(x.float(), y.float())
    loader = DataLoader(ds, batch_size=int(os.environ.get("NEUROVLM_BASELINE_MLP_BATCH_SIZE", "512")), shuffle=True)
    model.train()
    for _ in range(MLP_EPOCHS):
        for xb, yb in loader:
            xb = xb.to(device); yb = yb.to(device)
            pred = model(xb)
            loss = F.mse_loss(pred, yb)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
    model.eval()
    for p in model.parameters():
        p.requires_grad_(False)
    return model


@torch.no_grad()
def evaluate_baseline_for_reg(reg: dict[str, Any], baseline_name: str, ae, ae_arch, stage3_brain, stage3_text, text_cache, pred_fn, device: torch.device) -> dict[str, Any]:
    test_ds = make_domain_dataset(reg["test_manifest"], reg["domain"])
    loader = DataLoader(test_ds, batch_size=BASELINE_BATCH_SIZE, shuffle=False, collate_fn=lambda b: collate_primary(b, ae_arch["target_shape"]))
    gen_embs, text_embs, spatial_rows = [], [], []
    offset = 0
    for batch in loader:
        raw_cpu = stack_text_cache(text_cache, batch["texts"])
        raw = raw_cpu.to(device)
        target = batch["volume"].to(device)
        pred = pred_fn(batch, raw_cpu, offset).to(device)
        gen_embs.append(stage3_brain(pred.float().clamp(0.0, 1.0)).cpu())
        text_embs.append(stage3_text(raw.float()).cpu())
        spatial_rows.append(generation_metrics(pred.detach().cpu(), target.detach().cpu(), include_voxel_auroc=False))
        offset += len(batch["texts"])
    gen_emb = torch.cat(gen_embs, dim=0)
    text_emb = torch.cat(text_embs, dim=0)
    sim = F.normalize(text_emb, dim=1, eps=1e-8) @ F.normalize(gen_emb, dim=1, eps=1e-8).T
    auc = bidirectional_auc_from_similarity(sim)
    matched = sim.diag()
    shuffled = sim[torch.arange(sim.shape[0]), torch.randperm(sim.shape[0])] if sim.shape[0] > 1 else torch.tensor([float("nan")])
    spatial = {k: float(np.nanmean([row[k] for row in spatial_rows])) for k in spatial_rows[0]} if spatial_rows else {}
    return {
        "domain": reg["domain"],
        "branch_kind": reg["branch_kind"],
        "baseline": baseline_name,
        "n_test": int(sim.shape[0]),
        "text_to_generated_brain_normalized_auc": auc["text_to_brain_normalized_auc"],
        "generated_brain_to_text_normalized_auc": auc["brain_to_text_normalized_auc"],
        "mean_generation_normalized_recall_curve_auc": auc["mean_bidirectional_normalized_auc"],
        "matched_contrastive_cosine": float(matched.mean().item()),
        "shuffled_contrastive_cosine": float(shuffled.mean().item()),
        **spatial,
    }


def run_generation_baselines_for_reg(reg: dict[str, Any], device: torch.device) -> list[dict[str, Any]]:
    ae, ae_arch, _ = load_autoencoder(reg["ae_checkpoint"], device)
    stage3_brain, stage3_text, _ = load_stage3_models(reg["stage3_checkpoint"], device)
    text_cache = load_text_cache(reg["text_embedding_cache"])
    ref = build_reference_cache(reg, ae, ae_arch, stage3_text, text_cache, device)
    out = []

    def mean_pred(batch, raw_cpu, offset):
        return ref["mean_map"].unsqueeze(0).expand(len(batch["texts"]), -1, -1, -1, -1).contiguous()
    out.append(evaluate_baseline_for_reg(reg, "domain_mean_map", ae, ae_arch, stage3_brain, stage3_text, text_cache, mean_pred, device))

    train_stage3_text_n = F.normalize(ref["train_stage3_text"].float(), dim=1, eps=1e-8)
    def nn_indices(raw_cpu: torch.Tensor) -> torch.Tensor:
        q = F.normalize(stage3_text(raw_cpu.to(device)).cpu().float(), dim=1, eps=1e-8)
        return (q @ train_stage3_text_n.T).argmax(dim=1)

    def nn_map_pred(batch, raw_cpu, offset):
        idx = nn_indices(raw_cpu)
        return load_train_volumes_by_indices(ref["train_ds"], idx, ae_arch["target_shape"])
    out.append(evaluate_baseline_for_reg(reg, "nearest_neighbor_stage3_text_map", ae, ae_arch, stage3_brain, stage3_text, text_cache, nn_map_pred, device))

    def nn_latent_pred(batch, raw_cpu, offset):
        idx = nn_indices(raw_cpu)
        z = ref["train_latents"][idx].to(device)
        return ae.decoder(z).detach().cpu()
    out.append(evaluate_baseline_for_reg(reg, "nearest_neighbor_true_ae_latent_decoded", ae, ae_arch, stage3_brain, stage3_text, text_cache, nn_latent_pred, device))

    ridge_w = fit_ridge_closed_form(ref["train_raw_text"], ref["train_latents"], RIDGE_ALPHA)
    def ridge_pred(batch, raw_cpu, offset):
        z = predict_ridge(raw_cpu, ridge_w).to(device)
        return ae.decoder(z).detach().cpu()
    out.append(evaluate_baseline_for_reg(reg, "ridge_text_to_ae_latent", ae, ae_arch, stage3_brain, stage3_text, text_cache, ridge_pred, device))

    mlp = train_diagnostic_mlp(ref["train_raw_text"], ref["train_latents"], device)
    def mlp_pred(batch, raw_cpu, offset):
        z = mlp(raw_cpu.to(device))
        return ae.decoder(z).detach().cpu()
    mlp_row = evaluate_baseline_for_reg(reg, "diagnostic_mlp_768_512_384", ae, ae_arch, stage3_brain, stage3_text, text_cache, mlp_pred, device)
    mlp_row["mlp_epochs"] = MLP_EPOCHS
    mlp_row["mlp_lr"] = MLP_LR
    out.append(mlp_row)
    return out


if RUN_LATENT_DIAGNOSTICS:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    for reg in SOURCE_RUN_REGISTRY:
        try:
            rows, summary = run_latent_diagnostics_for_reg(reg, device)
            latent_rows.extend(rows)
            latent_summary_rows.append(summary)
            print("latent diagnostics complete", reg["domain"], reg["branch_kind"], summary["latent_mse_mean"])
        except Exception as exc:
            err = {"domain": reg.get("domain"), "branch_kind": reg.get("branch_kind"), "error": repr(exc)}
            latent_errors.append(err)
            print("latent diagnostics failed", err)
    write_csv(EVAL_ROOT / "06_latent_diagnostics/latent_distribution_diagnostics.csv", latent_rows)
    write_csv(EVAL_ROOT / "06_latent_diagnostics/latent_distribution_summary.csv", latent_summary_rows)
    write_json(EVAL_ROOT / "06_latent_diagnostics/latent_distribution_summary.json", {"status": "completed" if latent_rows else "empty_output", "rows": len(latent_rows), "summaries": latent_summary_rows, "errors": latent_errors})
else:
    write_csv(EVAL_ROOT / "06_latent_diagnostics/latent_distribution_diagnostics.csv", [])
    write_json(EVAL_ROOT / "06_latent_diagnostics/latent_distribution_summary.json", {"status": "skipped_by_flag", "run_flag": "NEUROVLM_RUN_LATENT_DIAGNOSTICS=1", "errors": []})

if RUN_GENERATION_BASELINES:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    for reg in SOURCE_RUN_REGISTRY:
        try:
            rows = run_generation_baselines_for_reg(reg, device)
            baseline_rows.extend(rows)
            print("generation baselines complete", reg["domain"], reg["branch_kind"], len(rows))
        except Exception as exc:
            err = {"domain": reg.get("domain"), "branch_kind": reg.get("branch_kind"), "error": repr(exc)}
            baseline_errors.append(err)
            print("generation baselines failed", err)
    write_csv(EVAL_ROOT / "07_generation_baselines/stage4_generation_baselines.csv", baseline_rows)
    write_json(EVAL_ROOT / "07_generation_baselines/stage4_generation_baselines_status.json", {"status": "completed" if baseline_rows else "empty_output", "rows": len(baseline_rows), "errors": baseline_errors, "mlp_epochs": MLP_EPOCHS, "ridge_alpha": RIDGE_ALPHA, "max_train_examples": MAX_BASELINE_TRAIN_EXAMPLES})
else:
    write_csv(EVAL_ROOT / "07_generation_baselines/stage4_generation_baselines.csv", [])
    write_json(EVAL_ROOT / "07_generation_baselines/stage4_generation_baselines_status.json", {"status": "skipped_by_flag", "run_flag": "NEUROVLM_RUN_GENERATION_BASELINES=1", "errors": []})


In [ ]:
# Final summary and reliable run status

def csv_row_count(path: str | Path) -> int:
    path = Path(path)
    if not path.exists() or path.stat().st_size == 0:
        return 0
    try:
        return max(0, len(pd.read_csv(path)))
    except Exception:
        return 0


def section_status(path: str | Path, *, required_columns: list[str] | None = None, status_json: str | Path | None = None) -> dict[str, Any]:
    path = Path(path)
    rows = csv_row_count(path) if path.suffix == ".csv" else None
    status = "completed"
    problems = []
    if not path.exists():
        status = "missing"
        problems.append("path_missing")
    elif path.suffix == ".csv" and rows == 0:
        status = "empty_output"
        problems.append("zero_rows")
    if required_columns and path.exists() and path.suffix == ".csv" and rows:
        cols = set(pd.read_csv(path, nrows=1).columns)
        missing = [c for c in required_columns if c not in cols]
        if missing:
            status = "failed"
            problems.append({"missing_columns": missing})
    if status_json and Path(status_json).exists():
        payload = json.loads(Path(status_json).read_text())
        if payload.get("status") in {"skipped_by_flag", "scaffold_only", "failed", "empty_output"}:
            status = payload.get("status")
        if payload.get("errors"):
            problems.append({"errors": payload.get("errors")})
            if status == "completed":
                status = "completed_with_errors"
    return {"path": str(path), "exists": path.exists(), "row_count": rows, "status": status, "problems": problems}

run_status = {
    "stage3_auc": section_status(EVAL_ROOT / "01_stage3_auc/stage3_normalized_auc_all_runs.csv", required_columns=["mean_bidirectional_normalized_auc"]),
    "legacy_stage4_checkpoint_eval": section_status(EVAL_ROOT / "02_existing_stage4_checkpoint_eval/stage4_checkpoint_comparison_all_runs.csv", required_columns=["mean_generation_normalized_recall_curve_auc"]),
    "pairing_audit": section_status(EVAL_ROOT / "04_pairing_audit/text_pairing_audit.csv", required_columns=["completed_experiment_policy"]),
    "text_cache_audit": section_status(EVAL_ROOT / "03_text_cache_audit/text_embedding_norm_audit.csv", required_columns=["norm_mean"]),
    "architecture_audit": section_status(EVAL_ROOT / "05_architecture_audit/existing_stage4_architecture_audit.csv", required_columns=["improperly_reused_stage3_contrastive_projector"]),
    "latent_diagnostics": section_status(
        EVAL_ROOT / "06_latent_diagnostics/latent_distribution_diagnostics.csv",
        required_columns=["latent_mse", "latent_cosine", "nearest_real_latent_distance"],
        status_json=EVAL_ROOT / "06_latent_diagnostics/latent_distribution_summary.json",
    ),
    "generation_baselines": section_status(
        EVAL_ROOT / "07_generation_baselines/stage4_generation_baselines.csv",
        required_columns=["baseline", "mean_generation_normalized_recall_curve_auc"],
        status_json=EVAL_ROOT / "07_generation_baselines/stage4_generation_baselines_status.json",
    ),
    "selected_existing_stage4_checkpoints_json": {"path": str(EVAL_ROOT / "02_existing_stage4_checkpoint_eval/selected_existing_stage4_checkpoints.json"), "exists": (EVAL_ROOT / "02_existing_stage4_checkpoint_eval/selected_existing_stage4_checkpoints.json").exists()},
    "reencode_stage4_text": bool(REENCODE_STAGE4_TEXT),
}
write_json(EVAL_ROOT / "00_metadata/run_status.json", run_status)
summary_rows = []
if "stage3_rows" in globals() and stage3_rows:
    summary_rows.extend({"section": "stage3", **r} for r in stage3_rows)
if "stage4_rows" in globals() and stage4_rows:
    summary_rows.extend({"section": "legacy_stage4", **r} for r in stage4_rows)
if "latent_summary_rows" in globals() and latent_summary_rows:
    summary_rows.extend({"section": "latent_diagnostics", **r} for r in latent_summary_rows)
if "baseline_rows" in globals() and baseline_rows:
    summary_rows.extend({"section": "generation_baseline", **r} for r in baseline_rows)
write_csv(EVAL_ROOT / "08_final_summary/existing_stage3_stage4_summary.csv", summary_rows)
(EVAL_ROOT / "08_final_summary/README_WHAT_TO_LOOK_AT.md").write_text("""# What To Look At

1. `01_stage3_auc/stage3_normalized_auc_all_runs.csv` is the primary Stage 3 table. Use mean bidirectional normalized recall-curve AUC, not fixed recall@K, for conclusions.
2. `02_existing_stage4_checkpoint_eval/stage4_checkpoint_comparison_all_runs.csv` ranks legacy Stage 4 checkpoints by mean generation normalized recall-curve AUC.
3. `04_pairing_audit/map_to_primary_text_manifest.csv` records the exact primary text used for every map and confirms the single-positive policy.
4. `03_text_cache_audit/reencoding_decision.json` says whether corrected Stage 4 should use a new SPECTER2 demeaned/unit-normalized generation cache.
5. `05_architecture_audit/existing_stage4_architecture_audit.csv` flags legacy Stage 4 checkpoints that reused or partially initialized from Stage 3 contrastive projection tensors.
6. `06_latent_diagnostics/latent_distribution_summary.json` tests whether text-derived latents are out-of-distribution for the frozen AE decoder.
7. `07_generation_baselines/stage4_generation_baselines.csv` reports non-neural/simple generation baselines using normalized generation recall-curve AUC.
8. `00_metadata/run_status.json` is authoritative for completed/skipped/empty/failed section status.
""")
run_status
